In [1]:
import numpy as np
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister
from qiskit.circuit.library import ModularAdderGate

def create_braidGate(strand_count, braid_sequence, k):
    '''
    A function that returns the gate that executes the braid representation of braid_sequence in B_(strand_count).
    k is a parameter relevant to approximating the Jones polynomial of said braid at k-th root of unity.
    braid_sequence is a list of nonzero integers. Negative integers denote inverse of the respective generator. List is read left to right.
    Reference: "Aharonov, et al - A Polynomial Quantum Algorithm for Approximating the Jones Polynomial"
    '''
    # Pre-computation
    lam = [0] + [np.sin(np.pi*v/k) for v in range(1,k)] + [0]
    A = (0+1j)*np.exp((0-1j)*np.pi/(2*k))
    anc_size = int(np.ceil(np.log2(2*k)))
    
    # Creating gates that execute the representation of braid generators
    phi = [A*np.array([[0,0,0,0],
                        [0,lam[v+1]/lam[v],np.sqrt(lam[v+1]*lam[v-1])/lam[v],0],
                        [0,np.sqrt(lam[v+1]*lam[v-1])/lam[v],lam[v-1]/lam[v],0],
                        [0,0,0,0]]) + A**(-1)*np.identity(4) for v in range(1,k)]
    phiGates = []
    for i in range(len(phi)):
        phiCirc = QuantumCircuit(2)
        phiCirc.unitary(phi[i],[0,1])
        phiGates.append(phiCirc.to_gate(label=f"phi_{i}"))
    ctrl_phiGates = [U.control(anc_size) for U in phiGates]

    # Creating gates that execute the representation of inverse braid generators
    phi_inv = [A**(-1)*np.array([[0,0,0,0],
                        [0,lam[v+1]/lam[v],np.sqrt(lam[v+1]*lam[v-1])/lam[v],0],
                        [0,np.sqrt(lam[v+1]*lam[v-1])/lam[v],lam[v-1]/lam[v],0],
                        [0,0,0,0]]) + A*np.identity(4) for v in range(1,k)]
    phiGates_inv = []
    for i in range(len(phi_inv)):
        phiCirc = QuantumCircuit(2)
        phiCirc.unitary(phi_inv[i],[0,1])
        phiGates_inv.append(phiCirc.to_gate(label=f"phi_inv_{i}"))
    ctrl_phiGates_inv = [U.control(anc_size) for U in phiGates_inv]

    #Creating the quantum circuit that will be turned into a gate
    path = QuantumRegister(strand_count,'path')
    anc = QuantumRegister(anc_size, 'anc')
    const = QuantumRegister(anc_size,'const')
    circ = QuantumCircuit(path, anc, const)

    #Creating a gate that "walks" and "unwalks" the ancilla
    adder = ModularAdderGate(anc_size)
    ctrl_adder = adder.control(1)

    ctrl_walk = QuantumRegister(1,'ctrl')
    anc_walk = QuantumRegister(anc_size, 'anc')
    const_walk = QuantumRegister(anc_size,'const')
    walk = QuantumCircuit(ctrl_walk, anc_walk, const_walk)

    walk.x(const_walk[0])
    walk.append(ctrl_adder, [ctrl_walk] + list(const_walk) + list(anc_walk))
    walk.x(const_walk[0])
    
    walk.x(const_walk[:])
    walk.x(ctrl_walk)
    walk.append(ctrl_adder, [ctrl_walk] + list(const_walk) + list(anc_walk))
    walk.x(ctrl_walk)
    walk.x(const_walk[:])
    

    walkGate = walk.to_gate()
    walkGate.label = 'walk'
    unwalkGate = walk.to_gate().inverse()
    unwalkGate.label = 'unwalk'
    #reversed(range(strand_count - (abs(gen)-1), strand_count))
    #Executing each braid generator
    for gen in braid_sequence:
        
        #Walk gen-1 steps
        for i in range(abs(gen)-1):
                circ.append(walkGate, [path[i]] + list(anc) + list(const))
        
        if gen > 0:
            #Multiplexing: if ancilla is equal to v, then execute Phi_v
            for v in range(1, k):
                zero_bits = [i for i in range(anc_size) if not (v >> i) & 1] # For each vertex v, find the index of 0s in a binary expansion of v
                circ.x(anc[zero_bits]) # Flip all the qubits in zero_bits to 1. This way, the multi-controlled phiGates only activates when the ancilla is precisely v
                circ.append(ctrl_phiGates[v-1], list(anc) + [path[gen - 1], path[gen]]) # Execute Phi_v on the i-th and (i+1)-th qubit with ancilla as controls
                circ.x(anc[zero_bits]) # Undo the flip to prepare for the next v
        
        elif gen < 0:
            #Multiplexing: if ancilla is equal to v, then execute Phi_inv_v
            for v in range(1, k):
                zero_bits = [i for i in range(anc_size) if not (v >> i) & 1] # For each vertex v, find the index of 0s in a binary expansion of v
                circ.x(anc[zero_bits]) # Flip all the qubits in zero_bits to 1. This way, the multi-controlled phiGates only activates when the ancilla is precisely v
                circ.append(ctrl_phiGates_inv[v-1], list(anc) + [path[abs(gen) - 1], path[abs(gen)]]) # Execute Phi_v on the i-th and (i+1)-th qubit with ancilla as controls
                circ.x(anc[zero_bits]) # Undo the flip to prepare for the next v
        
        else:
            return print("Error: invalid braid generator")
        
        #Unwalk gen-1 steps
        for i in reversed(range(abs(gen)-1)):
                circ.append(unwalkGate, [path[i]] + list(anc) + list(const))
                
    return circ.to_gate(label='braid')
    

In [ ]:
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit.quantum_info import SparsePauliOp
from qiskit import transpile

def jonespoly(strand_count, braid, k, shot_count):
    braidGate = create_braidGate(strand_count, braid, k)
    # Pre computation
    A = (0+1j)*np.exp((0-1j)*np.pi/(2*k))
    d = -A**2-A**(-2)
    anc_size = int(np.ceil(np.log2(2*k)))

    # Creating Hadamard test for the real component

    ctrl_braidGate = braidGate.control(1)

    had = QuantumRegister(1, 'had')
    path = QuantumRegister(strand_count,'path')
    anc = QuantumRegister(anc_size, 'anc')
    const = QuantumRegister(anc_size,'const')
    meas = ClassicalRegister(1, 'meas')
    
    reHad_circ = QuantumCircuit(had, path, anc, const, meas)
    reHad_circ.h(had[0])
    reHad_circ.x(path[[i for i in range(strand_count) if i % 2  == 0]])
    reHad_circ.x(anc[0])
    reHad_circ.append(ctrl_braidGate, [had[0]] + list(path) + list(anc) + list(const))
    reHad_circ.h(had[0])
    reHad_circ.measure(had[0],meas[0])

    # Creating Hadamard test for the imaginary component
    imHad_circ = QuantumCircuit(had, path, anc, const, meas)
    imHad_circ.h(had[0])
    imHad_circ.sdg(had[0])
    imHad_circ.x(path[[i for i in range(strand_count) if i % 2  == 0]])
    imHad_circ.x(anc[0])
    imHad_circ.append(ctrl_braidGate, [had[0]] + list(path) + list(anc) + list(const))
    imHad_circ.h(had[0])
    imHad_circ.measure(had[0],meas[0])
    
    sampler = Sampler()
    print(sampler.options.twirling.enable)
    reTrans = transpile(reHad_circ)
    imTrans = transpile(imHad_circ)
    
    job = sampler.run([reTrans, imTrans], shots=shot_count)
    reResult, imResult = job.result()
    reData, imData = reResult.data.meas, imResult.data.meas
    reCounts,imCounts = reData.get_counts(), imData.get_counts()

    if '0' not in reCounts:
        reJones = (0 - reCounts['1'])/shot_count
    elif '1' not in reCounts:
        reJones = (reCounts['0'] - 0)/shot_count
    else:
        reJones = (reCounts['0'] - reCounts['1'])/shot_count
    if '0' not in imCounts:
        imJones = (0 - imCounts['1'])/shot_count
    elif '1' not in imCounts:
        imJones = (imCounts['0'] - 0)/shot_count
    else:
        imJones = (imCounts['0'] - imCounts['1'])/shot_count

    inner = complex(round(reJones,5), round(imJones,5))

    #Commented out due to incorrect implementation of writhe of a plat closure. Unsure of how to perform it correctly. The output is a scalar multiple of the Jones polynomial for a link that is not oriented.
    # pos = [i for i in braid if i > 0]
    # neg = [i for i in braid if i < 0]
    # writhe = len(pos)-len(neg) 
    # jones = (-A)**(-3*writhe)*d**(strand_count/2-1)*inner
    return inner

In [2]:
shot_count = 2**10
jobs = [[2,[1],5],[2,[-1],5]]
for job in jobs:
    jones = jonespoly(job[0],job[1],job[2],shot_count)
    print(f"Braid {job[1]} in B{job[0]} inner product value at k={job[2]}: {jones}.")

NameError: name 'jonespoly' is not defined